In [3]:
import ast
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

# 1. Reproducibility
SEED = 7

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 2. Locate Model Notebook
candidates = [
    Path.cwd() / "02_electricity_model.ipynb",
    Path.cwd() / "experiments" / "04_electricity"
    / "02_electricity_model.ipynb",
    Path.cwd().parent / "experiments" / "04_electricity"
    / "02_electricity_model.ipynb",
]

MODEL_NOTEBOOK = next(
    (p.resolve() for p in candidates if p.is_file()),
    None
)

if MODEL_NOTEBOOK is None:
    raise FileNotFoundError(
        "Cannot find 02_electricity_model.ipynb"
    )

# 3. Read Notebook
with open(MODEL_NOTEBOOK, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# 4. Required Classes
required_classes = {
    "StrictStableKoopmanOperator",
    "SinCosPosEnc",
    "PatchEmbed1D",
    "PatchTST_Backbone",
    "Koopformer_PatchTST",
    "SeriesDecomp",
    "SimpleAutoformer",
    "Koopformer_Autoformer",
    "InformerSparse",
    "Koopformer_Informer",
    "SimpleLSTMForecaster",
}

# 5. Required Functions
required_functions = {
    "orthogonalize",
    "create_windows",
    "compute_loss",
    "train_model",
    "evaluate_model",
    "create_model",
}

# 6. Extract Definitions
definitions = []

for cell in notebook["cells"]:

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    # Parse Python source into AST
    tree = ast.parse(source)

    for node in tree.body:

        if (
            isinstance(node, ast.ClassDef)
            and node.name in required_classes
        ) or (
            isinstance(node, ast.FunctionDef)
            and node.name in required_functions
        ):
            definitions.append(node)

# 7. Compile and Load Definitions
module = ast.Module(
    body=definitions,
    type_ignores=[]
)

ast.fix_missing_locations(module)

exec(
    compile(
        module,
        str(MODEL_NOTEBOOK),
        "exec"
    ),
    globals()
)

# 8. Check Imported Definitions
required_names = required_classes | required_functions

missing = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing, f"Missing definitions: {missing}"

print("=== ELECTRICITY TRAINING ===")

print("\nDevice:", DEVICE)
print("Model Notebook:", MODEL_NOTEBOOK)

print("\nImported classes:", len(required_classes))
print("Imported functions:", len(required_functions))

print("\nAll model definitions loaded successfully!")

=== ELECTRICITY TRAINING ===

Device: cuda
Model Notebook: C:\Users\Win 11 New\Desktop\pa gk\experiments\04_electricity\02_electricity_model.ipynb

Imported classes: 11
Imported functions: 6

All model definitions loaded successfully!


In [5]:
from itertools import product

# 1. Locate Electricity Dataset
DATA_PATH = (
    MODEL_NOTEBOOK.parent
    / "data"
    / "electricity_train_test.pt"
)

assert DATA_PATH.is_file(), f"Dataset not found: {DATA_PATH}"

# 2. Load Preprocessed Data
bundle = torch.load(
    DATA_PATH,
    map_location="cpu",
    weights_only=False
)

train_data = bundle["train"]
test_data = bundle["test"]

INPUT_DIM = train_data.shape[1]
CHANNELS = bundle["channels"]

# 3. Grid Search Parameters
PATCH_LENGTHS = [
    70, 80, 90, 100, 110, 120, 130
]

HORIZONS = [
    2, 4, 6, 8, 10, 12, 14, 16
]

MODELS = [
    "LSTM",
    "PatchTST",
    "Autoformer",
    "Informer"
]

# 4. Model Hyperparameters
D_MODEL = 96
LSTM_HIDDEN_DIM = 48

EPOCHS = 4000
LEARNING_RATE = 3e-4
LYAP_WEIGHT = 0.1

# 5. Generate All Configurations
CONFIGURATIONS = list(
    product(
        PATCH_LENGTHS,
        HORIZONS,
        MODELS
    )
)

TOTAL_RUNS = len(CONFIGURATIONS)

# 6. Validate Configuration
assert train_data.shape == (2800, 6)
assert test_data.shape == (700, 6)

assert INPUT_DIM == 6
assert len(CHANNELS) == INPUT_DIM

assert torch.isfinite(train_data).all()
assert torch.isfinite(test_data).all()

assert TOTAL_RUNS == 224

assert bundle["data_hash"] == (
    "c024e2bb9460bf6e35d6f3353b82b6"
    "d90bee422366be0d6b3bb769a340eb0f0f"
)

# 7. Print Experiment Configuration
print("=== ELECTRICITY GRID SEARCH ===")

print("\nTrain:", train_data.shape)
print("Test:", test_data.shape)
print("Input dimension:", INPUT_DIM)

print("\nModels:", MODELS)
print("Patch lengths:", PATCH_LENGTHS)
print("Horizons:", HORIZONS)

print("\nKoopman d_model:", D_MODEL)
print("LSTM hidden dimension:", LSTM_HIDDEN_DIM)

print("\nTotal configurations:", TOTAL_RUNS)
print("Epochs per configuration:", EPOCHS)

print("\nLearning rate:", LEARNING_RATE)
print("Lyapunov weight:", LYAP_WEIGHT)

print("\nDataset SHA256:", bundle["data_hash"])

# 8. Preview First Configurations
print("\nFirst 8 configurations:")

for i, (p, h, model_name) in enumerate(
    CONFIGURATIONS[:8],
    start=1
):
    print(
        f"{i:3d}. "
        f"Model={model_name:10s} | "
        f"P={p} | H={h}"
    )

print("\nGrid Search configuration validated!")

=== ELECTRICITY GRID SEARCH ===

Train: torch.Size([2800, 6])
Test: torch.Size([700, 6])
Input dimension: 6

Models: ['LSTM', 'PatchTST', 'Autoformer', 'Informer']
Patch lengths: [70, 80, 90, 100, 110, 120, 130]
Horizons: [2, 4, 6, 8, 10, 12, 14, 16]

Koopman d_model: 96
LSTM hidden dimension: 48

Total configurations: 224
Epochs per configuration: 4000

Learning rate: 0.0003
Lyapunov weight: 0.1

Dataset SHA256: c024e2bb9460bf6e35d6f3353b82b6d90bee422366be0d6b3bb769a340eb0f0f

First 8 configurations:
  1. Model=LSTM       | P=70 | H=2
  2. Model=PatchTST   | P=70 | H=2
  3. Model=Autoformer | P=70 | H=2
  4. Model=Informer   | P=70 | H=2
  5. Model=LSTM       | P=70 | H=4
  6. Model=PatchTST   | P=70 | H=4
  7. Model=Autoformer | P=70 | H=4
  8. Model=Informer   | P=70 | H=4

Grid Search configuration validated!


In [6]:
# 1. Results Directory
RESULTS_DIR = (
    DATA_PATH.parent.parent
    / "results"
    / "electricity_grid_search_e4000"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# 2. Train One Configuration
def run_one_configuration(
    model_name,
    patch_len,
    horizon
):

    # Step 1: Checkpoint Path
    checkpoint_path = RESULTS_DIR / (
        f"{model_name.lower()}_p{patch_len}"
        f"_h{horizon}_d{D_MODEL}.pt"
    )

    # Skip completed configurations
    if checkpoint_path.exists():

        saved = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

        assert saved["data_hash"] == bundle["data_hash"]
        assert saved["epochs"] == EPOCHS
        assert saved["model"] == model_name
        assert saved["patch_len"] == patch_len
        assert saved["horizon"] == horizon
        assert saved["d_model"] == D_MODEL
        assert saved["input_dim"] == INPUT_DIM

        print(f"SKIP: {checkpoint_path.name}")

        return checkpoint_path

    print(
        f"\nSTART: {model_name} | "
        f"P={patch_len}, H={horizon}"
    )

    # Step 2: Sliding Windows
    X_train, y_train = create_windows(
        train_data,
        patch_len,
        horizon
    )

    X_test, y_test = create_windows(
        test_data,
        patch_len,
        horizon
    )

    print("X_train:", X_train.shape)
    print("y_train:", y_train.shape)

    print("X_test:", X_test.shape)
    print("y_test:", y_test.shape)

    # Step 3: Initialize Model
    model = create_model(
        model_name=model_name,
        patch_len=patch_len,
        horizon=horizon
    )

    # Step 4: Lyapunov Weight
    lyap_weight = (
        0.0 if model_name == "LSTM"
        else LYAP_WEIGHT
    )

    # Step 5: Training
    result = train_model(
        model=model,
        X_train=X_train,
        y_train=y_train,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        lyap_weight=lyap_weight,
        verbose=True
    )

    trained_model = result["model"]

    # Step 6: Evaluation
    train_metrics = evaluate_model(
        trained_model,
        X_train,
        y_train
    )

    test_metrics = evaluate_model(
        trained_model,
        X_test,
        y_test
    )

    # Step 7: Build Checkpoint
    checkpoint = {

        # Model Information
        "model": model_name,

        "model_state_dict": {
            k: v.detach().cpu().clone()
            for k, v in trained_model.state_dict().items()
        },

        # Train Predictions
        "pred_train": train_metrics["predictions"],
        "truth_train": y_train.cpu(),

        # Test Predictions
        "pred_test": test_metrics["predictions"],
        "truth_test": y_test.cpu(),

        # Training Histories
        "loss": torch.tensor(result["loss"]),
        "eig": torch.tensor(result["eig"]),
        "mse_history": torch.tensor(result["mse"]),
        "lyap_history": torch.tensor(result["lyap"]),

        # Train Metrics
        "train_mse": train_metrics["MSE"],
        "train_mae": train_metrics["MAE"],

        # Test Metrics
        "test_mse": test_metrics["MSE"],
        "test_mae": test_metrics["MAE"],

        # Hyperparameters
        "patch_len": patch_len,
        "horizon": horizon,
        "d_model": D_MODEL,
        "lstm_hidden_dim": LSTM_HIDDEN_DIM,
        "input_dim": INPUT_DIM,

        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "lyap_weight": lyap_weight,

        # Dataset Metadata
        "channels": CHANNELS,
        "data_hash": bundle["data_hash"],
    }

    # Step 8: Atomic Checkpoint Save
    temp_path = checkpoint_path.with_suffix(
        ".pt.tmp"
    )

    torch.save(checkpoint, temp_path)

    temp_path.replace(checkpoint_path)

    # Step 9: Print Results
    print(
        f"\nDONE: {model_name} | "
        f"P={patch_len}, H={horizon}"
    )

    print(
        f"Train MSE: {train_metrics['MSE']:.6f} | "
        f"Train MAE: {train_metrics['MAE']:.6f}"
    )

    print(
        f"Test MSE: {test_metrics['MSE']:.6f} | "
        f"Test MAE: {test_metrics['MAE']:.6f}"
    )

    print("Saved:", checkpoint_path)

    # Step 10: Release Memory
    del trained_model, model, result
    del X_train, y_train, X_test, y_test

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return checkpoint_path


print("Electricity training function defined successfully.")
print("Results directory:", RESULTS_DIR)

Electricity training function defined successfully.
Results directory: C:\Users\Win 11 New\Desktop\pa gk\experiments\04_electricity\results\electricity_grid_search_e4000


In [7]:
import os
from itertools import product

# 1. Metrics Configuration
METRICS_PATH = RESULTS_DIR / "metrics.csv"

METRIC_COLUMNS = [
    "Model",
    "PatchLen",
    "Horizon",
    "d_model",
    "Set",
    "Epochs",
    "MSE",
    "MAE"
]

CONFIGURATIONS = list(
    product(PATCH_LENGTHS, HORIZONS, MODELS)
)

assert len(CONFIGURATIONS) == 224


# 2. Checkpoint Path
def checkpoint_file(model_name, patch_len, horizon):

    return RESULTS_DIR / (
        f"{model_name.lower()}_p{patch_len}"
        f"_h{horizon}_d{D_MODEL}.pt"
    )


# 3. Collect Metrics from Checkpoints
def collect_metrics():

    rows = []
    completed = 0

    for patch_len, horizon, model_name in CONFIGURATIONS:

        path = checkpoint_file(
            model_name,
            patch_len,
            horizon
        )

        # Bỏ qua cấu hình chưa huấn luyện
        if not path.exists():
            continue

        # Load checkpoint
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        # Kiểm tra checkpoint thuộc đúng thực nghiệm
        assert ckpt["data_hash"] == bundle["data_hash"]
        assert ckpt["epochs"] == EPOCHS
        assert ckpt["model"] == model_name
        assert ckpt["patch_len"] == patch_len
        assert ckpt["horizon"] == horizon
        assert ckpt["d_model"] == D_MODEL
        assert ckpt["input_dim"] == INPUT_DIM
        assert ckpt["lstm_hidden_dim"] == LSTM_HIDDEN_DIM

        completed += 1

        # Mỗi checkpoint có hai bộ metrics: Train và Test
        for dataset in ["Train", "Test"]:

            prefix = dataset.lower()

            rows.append({
                "Model": model_name,
                "PatchLen": patch_len,
                "Horizon": horizon,
                "d_model": D_MODEL,
                "Set": dataset,
                "Epochs": EPOCHS,
                "MSE": ckpt[f"{prefix}_mse"],
                "MAE": ckpt[f"{prefix}_mae"]
            })

    # Tạo DataFrame
    metrics_df = pd.DataFrame(
        rows,
        columns=METRIC_COLUMNS
    )

    # Lưu metrics.csv an toàn
    temp_path = METRICS_PATH.with_suffix(
        ".csv.tmp"
    )

    metrics_df.to_csv(
        temp_path,
        index=False
    )

    os.replace(
        temp_path,
        METRICS_PATH
    )

    return completed, metrics_df


# 4. Grid Search Function
def run_grid_search(max_new_runs=None):

    # Kiểm tra những cấu hình đã hoàn thành
    completed, _ = collect_metrics()

    print("=== ELECTRICITY GRID SEARCH ===")
    print("Total configurations:", len(CONFIGURATIONS))
    print("Already completed:", completed)
    print("Epochs per configuration:", EPOCHS)
    print("Device:", DEVICE)

    new_runs = 0

    # Duyệt toàn bộ 224 cấu hình
    for index, (patch_len, horizon, model_name) in enumerate(
        CONFIGURATIONS,
        start=1
    ):

        path = checkpoint_file(
            model_name,
            patch_len,
            horizon
        )

        # Skip nếu đã hoàn thành
        if path.exists():

            print(
                f"SKIP [{index}/224] {path.name}"
            )

            continue

        # Giới hạn số lượt huấn luyện mới
        if (
            max_new_runs is not None
            and new_runs >= max_new_runs
        ):
            break

        print(
            f"\n[{index}/224] "
            f"{model_name} | "
            f"P={patch_len}, H={horizon}"
        )

        # Huấn luyện một cấu hình
        run_one_configuration(
            model_name=model_name,
            patch_len=patch_len,
            horizon=horizon
        )

        new_runs += 1

        # Cập nhật metrics sau mỗi lượt huấn luyện
        completed, metrics_df = collect_metrics()

        print(
            f"PROGRESS: {completed}/224 | "
            f"Metrics rows: {len(metrics_df)}"
        )

    # Tổng hợp kết quả cuối
    completed, metrics_df = collect_metrics()

    print("\n=== GRID SEARCH STATUS ===")
    print("Completed:", completed, "/224")
    print("New runs:", new_runs)
    print("Metrics rows:", len(metrics_df))
    print("Results directory:", RESULTS_DIR)

    return metrics_df


# 5. Check Definitions
print("Grid Search functions defined successfully.")
print("Total configurations:", len(CONFIGURATIONS))
print("Metrics path:", METRICS_PATH)

Grid Search functions defined successfully.
Total configurations: 224
Metrics path: C:\Users\Win 11 New\Desktop\pa gk\experiments\04_electricity\results\electricity_grid_search_e4000\metrics.csv


In [8]:
import ast
import json
from pathlib import Path

# 1. File Paths
PROJECT_DIR = MODEL_NOTEBOOK.parent.resolve()

TRAIN_NOTEBOOK = (
    PROJECT_DIR / "03_electricity_training.ipynb"
)

OUTPUT_SCRIPT = (
    PROJECT_DIR / "train_electricity.py"
)

assert MODEL_NOTEBOOK.is_file()
assert TRAIN_NOTEBOOK.is_file(), TRAIN_NOTEBOOK


# 2. Definitions from Model Notebook
model_definitions = [
    "orthogonalize",
    "StrictStableKoopmanOperator",
    "SinCosPosEnc",
    "PatchEmbed1D",
    "PatchTST_Backbone",
    "Koopformer_PatchTST",
    "SeriesDecomp",
    "SimpleAutoformer",
    "Koopformer_Autoformer",
    "InformerSparse",
    "Koopformer_Informer",
    "SimpleLSTMForecaster",
    "create_windows",
    "compute_loss",
    "train_model",
    "evaluate_model",
    "create_model",
]


# 3. Definitions from Training Notebook
training_definitions = [
    "run_one_configuration",
    "checkpoint_file",
    "collect_metrics",
    "run_grid_search",
]


# 4. Extract Python Definitions
def extract_definitions(notebook_path, names):

    with open(notebook_path, encoding="utf-8") as f:
        notebook = json.load(f)

    extracted = {}

    for cell in notebook["cells"]:

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        tree = ast.parse(source)

        for node in tree.body:

            if not isinstance(
                node,
                (ast.ClassDef, ast.FunctionDef)
            ):
                continue

            if node.name in names:

                if node.name in extracted:
                    raise ValueError(
                        f"Duplicate definition: {node.name}"
                    )

                extracted[node.name] = (
                    ast.get_source_segment(
                        source,
                        node
                    )
                )

    missing = set(names) - set(extracted)

    if missing:
        raise ValueError(
            f"Missing definitions: {missing}"
        )

    return "\n\n\n".join(
        extracted[name]
        for name in names
    )


# 5. Extract Model and Training Code
model_code = extract_definitions(
    MODEL_NOTEBOOK,
    model_definitions
)

training_code = extract_definitions(
    TRAIN_NOTEBOOK,
    training_definitions
)


# ==========================================
# 6. Standalone Script Header
# ==========================================
header = '''
import argparse
import os
import random

from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim


# Reproducibility
SEED = 7

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# Dataset
BASE_DIR = Path(__file__).resolve().parent

DATA_PATH = (
    BASE_DIR / "data" / "electricity_train_test.pt"
)

bundle = torch.load(
    DATA_PATH,
    map_location="cpu",
    weights_only=False
)

train_data = bundle["train"]
test_data = bundle["test"]

INPUT_DIM = train_data.shape[1]
CHANNELS = bundle["channels"]

assert train_data.shape == (2800, 6)
assert test_data.shape == (700, 6)
assert INPUT_DIM == 6


# Grid Search
PATCH_LENGTHS = [70, 80, 90, 100, 110, 120, 130]
HORIZONS = [2, 4, 6, 8, 10, 12, 14, 16]

MODELS = [
    "LSTM",
    "PatchTST",
    "Autoformer",
    "Informer"
]


# Hyperparameters
D_MODEL = 96
LSTM_HIDDEN_DIM = 48

EPOCHS = 4000
LEARNING_RATE = 3e-4
LYAP_WEIGHT = 0.1


# Results
RESULTS_DIR = (
    BASE_DIR
    / "results"
    / "electricity_grid_search_e4000"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METRICS_PATH = RESULTS_DIR / "metrics.csv"

METRIC_COLUMNS = [
    "Model", "PatchLen", "Horizon",
    "d_model", "Set", "Epochs",
    "MSE", "MAE"
]

CONFIGURATIONS = list(
    product(
        PATCH_LENGTHS,
        HORIZONS,
        MODELS
    )
)

assert len(CONFIGURATIONS) == 224
'''


# ==========================================
# 7. Standalone Script CLI
# ==========================================
footer = '''
if __name__ == "__main__":

    parser = argparse.ArgumentParser(
        description="Electricity DeepKoopFormer Grid Search"
    )

    parser.add_argument(
        "--limit",
        type=int,
        default=1,
        help="Maximum number of new configurations to train"
    )

    parser.add_argument(
        "--dry-run",
        action="store_true",
        help="Validate configuration without training"
    )

    args = parser.parse_args()

    if args.limit < 1:
        parser.error("--limit must be >= 1")

    print("=== ELECTRICITY GRID SEARCH ===")

    print("Device:", DEVICE)

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    )

    print("Train shape:", tuple(train_data.shape))
    print("Test shape:", tuple(test_data.shape))

    print("Total configurations:", len(CONFIGURATIONS))
    print("Epochs:", EPOCHS)

    print("Koopman d_model:", D_MODEL)
    print("LSTM hidden dimension:", LSTM_HIDDEN_DIM)

    print("Dataset hash:", bundle["data_hash"])

    if args.dry_run:

        completed, metrics_df = collect_metrics()

        print("\\\\nDRY RUN")
        print("Completed:", completed)
        print("Remaining:", len(CONFIGURATIONS) - completed)
        print("Metrics rows:", len(metrics_df))
        print("No training performed.")

    else:

        run_grid_search(
            max_new_runs=args.limit
        )
'''


# ==========================================
# 8. Combine Script
# ==========================================
script = (
    header
    + "\n\n"
    + model_code
    + "\n\n"
    + training_code
    + "\n\n"
    + footer
)

# Kiểm tra cú pháp mà không huấn luyện
compile(
    script,
    str(OUTPUT_SCRIPT),
    "exec"
)

# Ghi file Python
OUTPUT_SCRIPT.write_text(
    script,
    encoding="utf-8"
)


# ==========================================
# 9. Verify Export
# ==========================================
assert OUTPUT_SCRIPT.is_file()

print("=== ELECTRICITY SCRIPT EXPORT ===")

print("\nScript created:", OUTPUT_SCRIPT)

print(
    "File size:",
    round(
        OUTPUT_SCRIPT.stat().st_size / 1024,
        2
    ),
    "KB"
)

print("\nModel definitions:", len(model_definitions))
print("Training definitions:", len(training_definitions))

print("\nTotal configurations:", len(CONFIGURATIONS))
print("Syntax check: PASSED")
print("No training performed.")

ValueError: Duplicate definition: PatchEmbed1D